# Dataset Comparison: V3 Original vs V4 New
## Side-by-side analysis of the training data that produced 0.740 AUC vs 0.576 AUC

V3 original dataset recovered from Hetzner server (Feb 3, 2026 training run).
V4 dataset built with extended date range through Mar 5, 2026.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

v3 = pd.read_csv('/tmp/v3_original_POINTS.csv')
v4 = pd.read_csv('nba/features/datasets/xl_v2_matchup_training_POINTS_2023_2025.csv')

print(f'V3: {v3.shape} — {v3["game_date"].min()} to {v3["game_date"].max()}')
print(f'V4: {v4.shape} — {v4["game_date"].min()} to {v4["game_date"].max()}')

In [ ]:
# Column comparison
v3_cols = set(v3.columns)
v4_cols = set(v4.columns)

shared = v3_cols & v4_cols
v3_only = v3_cols - v4_cols
v4_only = v4_cols - v3_cols

print(f'Shared columns: {len(shared)}')
print(f'V3-only columns: {len(v3_only)}')
if v3_only:
    for c in sorted(v3_only):
        print(f'  {c}')
print(f'V4-only columns: {len(v4_only)}')
for c in sorted(v4_only):
    # Check if populated
    if c in v4.columns:
        pop = (v4[c] != 0).mean() * 100 if v4[c].dtype != 'bool' else 100
        print(f'  {c}: {pop:.0f}% populated')

In [ ]:
# Compare shared features — are the distributions different?
diffs = []
for c in sorted(shared):
    if c in ['player_name','game_date','stat_type','opponent_team','source','split','game_id','player_id','game_time']:
        continue
    v3_vals = v3[c] if v3[c].dtype != 'bool' else v3[c].astype(int)
    v4_vals = v4[c] if v4[c].dtype != 'bool' else v4[c].astype(int)
    
    if v3_vals.dtype not in [np.float64, np.int64, float, int]:
        continue
    
    v3_mean = v3_vals.mean()
    v4_mean = v4_vals.mean()
    v3_std = v3_vals.std()
    v4_std = v4_vals.std()
    v3_nz = (v3_vals != 0).mean()
    v4_nz = (v4_vals != 0).mean()
    
    diffs.append({
        'feature': c,
        'v3_mean': round(v3_mean, 3),
        'v4_mean': round(v4_mean, 3),
        'mean_diff': round(v4_mean - v3_mean, 3),
        'v3_nonzero': round(v3_nz, 3),
        'v4_nonzero': round(v4_nz, 3),
        'nz_diff': round(v4_nz - v3_nz, 3),
    })

diff_df = pd.DataFrame(diffs)

# Biggest differences in non-zero rate
print('=== Biggest drops in population rate (V3 → V4) ===')
print(diff_df.sort_values('nz_diff').head(20).to_string(index=False))

In [ ]:
# Target variable comparison
for label, df_check in [('V3', v3), ('V4', v4)]:
    print(f'\n{label}:')
    if 'label' in df_check.columns:
        print(f'  label distribution: {df_check["label"].value_counts().to_dict()}')
        print(f'  label rate (P(over)): {df_check["label"].mean():.3f}')
    if 'actual_points' in df_check.columns:
        print(f'  actual_points: mean={df_check["actual_points"].mean():.1f}, zeros={(df_check["actual_points"]==0).sum()}')
    if 'line' in df_check.columns:
        print(f'  line: mean={df_check["line"].mean():.1f}')
    print(f'  is_home dtype: {df_check["is_home"].dtype}')

In [ ]:
# Visualize the key differences
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. H2H games
axes[0,0].hist(v3['h2h_games'], bins=20, alpha=0.6, label='V3', color='green')
axes[0,0].hist(v4['h2h_games'], bins=20, alpha=0.6, label='V4', color='red')
axes[0,0].set_title('h2h_games')
axes[0,0].legend()

# 2. H2H avg points
axes[0,1].hist(v3['h2h_avg_points'][v3['h2h_avg_points']>0], bins=20, alpha=0.6, label='V3', color='green')
axes[0,1].hist(v4['h2h_avg_points'][v4['h2h_avg_points']>0], bins=20, alpha=0.6, label='V4', color='red')
axes[0,1].set_title('h2h_avg_points (non-zero only)')
axes[0,1].legend()

# 3. is_home
v3_home = v3['is_home'].astype(int) if v3['is_home'].dtype != 'bool' else v3['is_home'].astype(int)
v4_home = v4['is_home'].astype(int) if v4['is_home'].dtype != 'bool' else v4['is_home'].astype(int)
axes[0,2].bar(['V3 Home','V3 Away','V4 Home','V4 Away'], 
              [v3_home.sum(), len(v3_home)-v3_home.sum(), v4_home.sum(), len(v4_home)-v4_home.sum()],
              color=['green','lightgreen','red','lightsalmon'])
axes[0,2].set_title('is_home distribution')

# 4. Line distribution
axes[1,0].hist(v3['line'], bins=30, alpha=0.6, label='V3', color='green')
axes[1,0].hist(v4['line'], bins=30, alpha=0.6, label='V4', color='red')
axes[1,0].set_title('Prop line distribution')
axes[1,0].legend()

# 5. Rows per month
v3_months = pd.to_datetime(v3['game_date']).dt.to_period('M').value_counts().sort_index()
v4_months = pd.to_datetime(v4['game_date']).dt.to_period('M').value_counts().sort_index()
all_months = sorted(set(v3_months.index) | set(v4_months.index))
x = range(len(all_months))
axes[1,1].bar([i-0.2 for i in x], [v3_months.get(m, 0) for m in all_months], width=0.4, label='V3', color='green', alpha=0.7)
axes[1,1].bar([i+0.2 for i in x], [v4_months.get(m, 0) for m in all_months], width=0.4, label='V4', color='red', alpha=0.7)
axes[1,1].set_xticks(x)
axes[1,1].set_xticklabels([str(m) for m in all_months], rotation=45, ha='right', fontsize=6)
axes[1,1].set_title('Rows per month')
axes[1,1].legend()

# 6. Feature count comparison
v3_usable = len([c for c in v3.select_dtypes(include='number').columns if v3[c].std() > 0])
v4_usable = len([c for c in v4.select_dtypes(include='number').columns if v4[c].std() > 0])
axes[1,2].bar(['V3 Total','V3 Usable','V4 Total','V4 Usable'], 
              [len(v3.columns), v3_usable, len(v4.columns), v4_usable],
              color=['green','darkgreen','red','darkred'])
axes[1,2].set_title('Feature counts')

plt.tight_layout()
plt.savefig('notebooks/dataset_comparison.png', dpi=150)
plt.show()